In [3]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

# 1. Load the Transcript Data
transcript_path = '../data/Amirim_Project_Submission/translated_podcast_transcript_filtered.csv'
df = pd.read_csv(transcript_path)

# 2. Initialize the Multilingual Contextual Model
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = AutoModel.from_pretrained('xlm-roberta-base')

# 3. Define the Embedding Function
def get_sentence_embedding(text):
    if pd.isna(text):
        return None
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    last_hidden_state = outputs.last_hidden_state
    sentence_embedding = torch.mean(last_hidden_state, dim=1).squeeze()
    return sentence_embedding.numpy().tolist()

# 4. Generate Embeddings for All Three Languages
print("Extracting English embeddings...")
df['en_contextual'] = df['en'].apply(get_sentence_embedding)

print("Extracting Hebrew embeddings...")
df['he_contextual'] = df['he'].apply(get_sentence_embedding)

print("Extracting Arabic embeddings...")
df['ar_contextual'] = df['ar'].apply(get_sentence_embedding)

# 5. Save the New Features
output_path = '../data/processed/contextual_podcast_embeddings.csv'
df.to_csv(output_path, index=False)
print(f"Success! Contextual embeddings saved to {output_path}")

Loading model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13099.79it/s]
XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting English embeddings...
Extracting Hebrew embeddings...
Extracting Arabic embeddings...
Success! Contextual embeddings saved to ../data/processed/contextual_podcast_embeddings.csv
